# Scientific ML lifecycle and graph evidence

A small CPU example of the existing VAFT lifecycle. Graph extraction is example-specific; scientific validity remains a domain decision. Optional PyTorch and Graphviz are used here. See `workflow/ml_research/README.md` for structured sources and GPU qualification.

In [ ]:
import numpy as np
import torch
from vaft.process import ml
from vaft.diagram.ml import extract_torch_graph
from vaft.process.ml.experiments import experiment_record
from vaft.plot.ml import learning_curves
from vaft.plot.renderers.lines import render_line_series
rng = np.random.default_rng(7)
x = rng.normal(size=(48, 3))
data = ml.build_dataset(x, np.repeat(np.arange(8), 6), targets=x[:, :1] * 2,
    spec=ml.DatasetSpec(name="example", task="regression", group_key="shot"))
split = ml.split_groups(data)
result = ml.train_model(data, split, ml.ModelSpec(architecture="mlp", task="example", hyperparameters={"hidden": [4]}),
    ml.TrainingConfig(epochs=3, batch_size=8))
ml.evaluate_model(result.artifact, data, split).metrics

In [ ]:
# This backend adapter reconstructs a copy; plot renderers never receive a module.
from vaft.process.ml._backends import get_backend
net, _ = get_backend("torch")._load(result.artifact.model_spec, result.artifact.state)
extraction = extract_torch_graph(net, (torch.zeros(2, 3),), name="VAFT MLP")
if extraction.supported:
    display(extraction.graph)
else:
    print(extraction.diagnostics)

In [ ]:
curves = learning_curves(result.history, keys=("train_loss", "validation_loss"))
fig, ax = render_line_series(curves)
record = experiment_record(result)
assert record["model_identity"]["state_sha256"] == result.artifact.state_sha256

Structured reference models and original/adapter physical prediction checks live in the NeurIPS opt-in research adapter. MAST inference remains label-free. This notebook performs no tracking upload, model publication or full research training.